In [1]:
LEAGUE = "brasileirao-serie-a"
SEASON = "2024"
ENVIRONMENT = "prd"


In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("gold") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://cvmc-minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.driver.memory", "4g") \
    .config("spark.driver.maxResultSize", "2g") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

print(f"Spark version: {spark.version}")
print("Configurado!")

Spark version: 3.5.0
Configurado!


In [3]:
import psycopg2

conn = psycopg2.connect(
    host="cvmc-postgres",
    port=5432,
    dbname="cvmc_data",
    user="airflow",
    password="airflow"
)
conn.autocommit = True
cur = conn.cursor()
cur.execute("CREATE SCHEMA IF NOT EXISTS gold;")
cur.close()
conn.close()
print("Schema gold garantido!")

Schema gold garantido!


In [4]:
# =====================
# PARÂMETROS DE EXECUÇÃO
# =====================
# LEAGUE = "brasileirao-serie-a"
# SEASON = "2026"
# ENVIRONMENT = "dev"

# =====================
# CONFIGURAÇÕES FIXAS
# =====================
BUCKET = "eng-prd-data-lake"
SILVER_PATH = f"s3a://{BUCKET}/{ENVIRONMENT}/silver/{LEAGUE}"

# Postgres
PG_URL = "jdbc:postgresql://cvmc-postgres:5432/cvmc_data?currentSchema=gold"
PG_PROPERTIES = {
    "user": "airflow",
    "password": "airflow",
    "driver": "org.postgresql.Driver"
}

print(f"League: {LEAGUE} | Season: {SEASON} | Env: {ENVIRONMENT}")
print(f"Silver: {SILVER_PATH}")
print(f"Postgres: {PG_URL}")

League: brasileirao-serie-a | Season: 2024 | Env: prd
Silver: s3a://eng-prd-data-lake/prd/silver/brasileirao-serie-a
Postgres: jdbc:postgresql://cvmc-postgres:5432/cvmc_data?currentSchema=gold


In [5]:
from pyspark.sql.functions import current_timestamp, from_utc_timestamp

def save_to_postgres(df, table_name, mode="overwrite"):
    print(f"Salvando {table_name} no Postgres...")
    df = df.withColumn(
        "dt_updated_at",
        from_utc_timestamp(current_timestamp(), "America/Sao_Paulo")
    )
    df.write.jdbc(
        url=PG_URL,
        table=table_name,
        mode=mode,
        properties=PG_PROPERTIES
    )
    print(f"[OK] {table_name} salvo! Registros: {df.count()}\n")

In [6]:
from pyspark.sql.functions import col

df_standing = spark.read.format("delta").load(f"{SILVER_PATH}/standing")

df_team_season = df_standing.select(
    col("season"),
    col("league"),
    col("record.teamId").alias("team_id"),
    col("record.teamName").alias("team_name"),
    col("record.teamSlug").alias("team_slug"),
    col("record.position").alias("position"),
    col("record.points").alias("points"),
    col("record.matchesPlayed").alias("matches_played"),
    col("record.wins").alias("wins"),
    col("record.draws").alias("draws"),
    col("record.losses").alias("losses"),
    col("record.goalsFor").alias("goals_for"),
    col("record.goalsAgainst").alias("goals_against"),
    col("record.goalDifference").alias("goal_difference"),
    col("record.shots").alias("shots"),
    col("record.shotsOnTarget").alias("shots_on_target"),
    col("record.shotsOffTarget").alias("shots_off_target"),
    col("record.expectedGoals").alias("expected_goals"),
    col("record.expectedAssists").alias("expected_assists"),
    col("record.possession").alias("possession"),
    col("record.passesFor").alias("passes_for"),
    col("record.passesAgainst").alias("passes_against"),
    col("record.yellowCards").alias("yellow_cards"),
    col("record.redCards").alias("red_cards"),
    col("record.fouls").alias("fouls"),
    col("record.cornersFor").alias("corners_for"),
    col("record.cornersAgainst").alias("corners_against"),
    col("record.tacklesFor").alias("tackles_for"),
    col("record.tacklesAgainst").alias("tackles_against"),
    col("record.interceptions").alias("interceptions"),
    col("record.bigChances").alias("big_chances"),
    col("record.xGFor").alias("xg_for"),
    col("record.xGAgainst").alias("xg_against"),
    col("record.keyPasses").alias("key_passes"),
    col("record.successfulDribbles").alias("successful_dribbles"),
    col("record.saves").alias("saves")
)

# df_team_season.show(5)
save_to_postgres(df_team_season, "obt_team_season_stats")

Salvando obt_team_season_stats no Postgres...
[OK] obt_team_season_stats salvo! Registros: 60



In [7]:
from pyspark.sql.functions import col, explode, lit, first
from functools import reduce
from pyspark.sql import DataFrame
from pyspark.sql.functions import year, from_unixtime

df_team_stats = spark.read.format("delta").load(f"{SILVER_PATH}/team_stats")

dfs = []
# Pegando apenas o event_type "all" para ficar mais leve e limpo!
for event_type in ["all"]:
    for key in [
        "accurateCross", "bigChanceCreated", "bigChanceMissed", "bigChanceScored",
        "expectedGoals", "shotsOnGoal", "shotsOffGoal", "totalShotsInsideBox", "totalShotsOutsideBox",
        "totalClearance", "dispossessed", "errorsLeadToGoal", "errorsLeadToShot",
        "fouls", "goalkeeperSaves", "interceptionWon", "totalTackle",
        "freeKicks", "goalKicks", "throwIns",
        "ballPossession", "offsides", "passes", "touchesInOppBox",
        "redCards", "yellowCards"
    ]:
        df_temp = df_team_stats.select(
            col("season"),
            col("league"),
            col("record.team.teamId").alias("team_id"),
            col("record.team.teamName").alias("team_name"),
            col("record.team.teamSlug").alias("team_slug"),
            lit(event_type).alias("event_type"),
            explode(col(f"record.stats.{event_type}.{key}")).alias("stat")
        ).select(
            "season", "league", "team_id", "team_name", "team_slug", "event_type",
            col("stat.statisticKey").alias("statistic_key"),
            col("stat.event_id"),
            col("stat.home_team_id"),
            col("stat.home_team_name"),
            col("stat.away_team_id"),
            col("stat.away_team_name"),
            col("stat.home_score"),
            col("stat.away_score"),
            col("stat.home_value").cast("double").alias("home_value"),
            col("stat.away_value").cast("double").alias("away_value"),
            col("stat.time_start_timestamp")
        )
        dfs.append(df_temp)

# Unindo tudo
df_team_match = reduce(DataFrame.unionByName, dfs)

df_team_match = df_team_match.withColumn(
    "season",
    year(from_unixtime(col("time_start_timestamp").cast("long"))).cast("string")
)

# Transformando as linhas em colunas e mantendo a chave única da partida
df_team_match_pivoted = df_team_match.groupBy(
    "season", "league", "team_id", "team_name", "team_slug", "event_type",
    "event_id", "home_team_id", "home_team_name", "away_team_id", "away_team_name",
    "home_score", "away_score", "time_start_timestamp"
).pivot("statistic_key").agg(
    first("home_value").alias("home"),
    first("away_value").alias("away")
)

print(f"Total registros na tabela OBT: {df_team_match_pivoted.count()}")

# Marcando aquele golaço e salvando no Postgres
save_to_postgres(df_team_match_pivoted, "obt_team_match_stats")

Total registros na tabela OBT: 1644
Salvando obt_team_match_stats no Postgres...
[OK] obt_team_match_stats salvo! Registros: 1644



In [8]:
from pyspark.sql.functions import col

df_players = spark.read.format("delta").load(f"{SILVER_PATH}/players")

df_player_season = df_players.select(
    col("season"),
    col("league"),
    col("record.players.id").alias("player_id"),
    col("record.players.name").alias("player_name"),
    col("record.players.slug").alias("player_slug"),
    col("record.players.position").alias("position"),
    col("record.teams.teamId").alias("team_id"),
    col("record.teams.teamName").alias("team_name"),
    col("record.teams.teamSlug").alias("team_slug"),
    col("record.player_statistics.appearances").cast("integer").alias("appearances"),
    col("record.player_statistics.minutesPlayedTotal").cast("integer").alias("minutes_played"),
    col("record.player_statistics.goalsTotal").cast("integer").alias("goals"),
    col("record.player_statistics.goalAssistTotal").cast("integer").alias("assists"),
    col("record.player_statistics.expectedGoalsTotal").cast("double").alias("expected_goals"),
    col("record.player_statistics.expectedAssistsTotal").cast("double").alias("expected_assists"),
    col("record.player_statistics.onTargetScoringAttemptTotal").cast("integer").alias("shots_on_target"),
    col("record.player_statistics.shotOffTargetTotal").cast("integer").alias("shots_off_target"),
    col("record.player_statistics.totalPassTotal").cast("integer").alias("passes"),
    col("record.player_statistics.accuratePassTotal").cast("integer").alias("accurate_passes"),
    col("record.player_statistics.keyPassTotal").cast("integer").alias("key_passes"),
    col("record.player_statistics.totalTackleTotal").cast("integer").alias("tackles"),
    col("record.player_statistics.interceptionWonTotal").cast("integer").alias("interceptions"),
    col("record.player_statistics.duelWonTotal").cast("integer").alias("duels_won"),
    col("record.player_statistics.duelLostTotal").cast("integer").alias("duels_lost"),
    col("record.player_statistics.yellowCardTotal").cast("integer").alias("yellow_cards"),
    col("record.player_statistics.redCardTotal").cast("integer").alias("red_cards"),
    col("record.player_statistics.ratingTotal").cast("double").alias("rating"),
    col("record.player_statistics.savesTotal").cast("integer").alias("saves"),
    col("record.player_statistics.totalClearanceTotal").cast("integer").alias("clearances"),
    col("record.player_statistics.foulsTotal").cast("integer").alias("fouls"),
    col("record.player_statistics.wasFouledTotal").cast("integer").alias("was_fouled")
)

# df_player_season.show(5)
save_to_postgres(df_player_season, "obt_player_season_stats")

Salvando obt_player_season_stats no Postgres...
[OK] obt_player_season_stats salvo! Registros: 1152



In [9]:
from pyspark.sql.functions import col, explode, lit

df_player_stats = spark.read.format("delta").load(f"{SILVER_PATH}/player_stats")

dfs = []
for location in ["both", "home", "away"]:
    df_temp = df_player_stats.select(
        col("season"),
        col("league"),
        col("record.team.teamId").alias("team_id"),
        col("record.team.teamName").alias("team_name"),
        col("record.team.teamSlug").alias("team_slug"),
        lit(location).alias("location"),
        explode(col(f"record.player_stats.{location}")).alias("player")
    ).select(
        "season", "league", "team_id", "team_name", "team_slug", "location",
        col("player.id").alias("player_id"),
        col("player.name").alias("player_name"),
        col("player.slug").alias("player_slug"),
        col("player.position").alias("position"),
        explode(col("player.stats")).alias("stat")
    ).select(
        "season", "league", "team_id", "team_name", "team_slug", "location",
        "player_id", "player_name", "player_slug", "position",
        col("stat.eventId").alias("event_id"),
        col("stat.minutesPlayed").alias("minutes_played"),
        col("stat.goals").alias("goals"),
        col("stat.goalAssist").alias("assists"),
        col("stat.expectedGoals").cast("double").alias("expected_goals"),
        col("stat.expectedAssists").cast("double").alias("expected_assists"),
        col("stat.onTargetScoringAttempt").alias("shots_on_target"),
        col("stat.shotOffTarget").alias("shots_off_target"),
        col("stat.totalPass").alias("passes"),
        col("stat.accuratePass").alias("accurate_passes"),
        col("stat.keyPass").alias("key_passes"),
        col("stat.totalTackle").alias("tackles"),
        col("stat.interceptionWon").alias("interceptions"),
        col("stat.duelWon").alias("duels_won"),
        col("stat.duelLost").alias("duels_lost"),
        col("stat.yellowCard").alias("yellow_cards"),
        col("stat.redCard").alias("red_cards"),
        col("stat.rating").cast("double").alias("rating"),
        col("stat.saves").alias("saves"),
        col("stat.totalClearance").alias("clearances"),
        col("stat.fouls").alias("fouls"),
        col("stat.wasFouled").alias("was_fouled"),
        col("stat.touches").alias("touches"),
        col("stat.xGxA").alias("xgxa")
    )
    dfs.append(df_temp)

from functools import reduce
from pyspark.sql import DataFrame

df_player_match = reduce(DataFrame.unionByName, dfs)
print(f"Total registros: {df_player_match.count()}")
# df_player_match.show(5)
save_to_postgres(df_player_match, "obt_player_match_stats")

Total registros: 75124
Salvando obt_player_match_stats no Postgres...
[OK] obt_player_match_stats salvo! Registros: 75124

